# Parts 3–4 · 00 Setup and access check

This notebook runs once before any download. It:

1. records the environment (Python and package versions, for the reproducibility checklist);
2. connects to WRDS with **your own** account;
3. checks access to every table parts 3–4 need and records their exact column names;
4. builds the trading calendar from CRSP;
5. rebuilds the provisional release clock on that calendar.

**Before running:** clone the repository, open this notebook inside it, and have Duo Mobile on your phone.

**Folder layout**

```
S08/3_4_surprise_returns/
  code/     common.py, clock.py; later download, surprise, returns, funnel
  cache/    raw WRDS pulls, one file per table and year (WRDS-derived: never in the submission)
  data/     built tables keyed by accession
  results/  tables and figures for the report
  logs/     access reports and run logs
```

In [3]:
%pip install -q wrds pandas_market_calendars pyarrow


In [ ]:
import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, SCRIPTS, NOV2_DATA,
                           PROJECT_DIR, STEP5_DIR, PART34_CODE, STEP5_CODE)

# Notebook paths.
MAHAUT_DATA = NOV2_DATA   # only used by the optional clock cell



for sub in ["cache", "data", "results", "logs"]:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(PART34_CODE))

missing = [f for f in ["common.py", "clock.py"] if not (PART34_CODE / f).exists()]
assert not missing, f"Missing code in {PROJECT_DIR / 'code'}: {missing}"

print("team folder:   ", TEAM_DIR)
print("project folder:", PROJECT_DIR)
print("code files:    ", sorted(os.listdir(PART34_CODE)))

In [ ]:
# Environment record: goes into the access report and, later, the README.
import json, platform, datetime as dt
import importlib.metadata as md

env = {"run_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version()}
for pkg in ["wrds", "pandas", "numpy", "sqlalchemy", "psycopg2-binary",
            "pandas_market_calendars", "pyarrow"]:
    try:
        env[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        env[pkg] = "not installed"
print(json.dumps(env, indent=2))

## Connect to WRDS

Use your own account: WRDS accounts are personal, and entitlements belong to the account, so Mahaut's check does not cover yours. You will be asked for your password; then approve the Duo push on your phone within a few minutes.

If asked whether to create a `.pgpass` file, answer **n**. Never put your password in the notebook or in Drive.

In [ ]:
import wrds

WRDS_USER = input("WRDS username: ").strip()
db = wrds.Connection(wrds_username=WRDS_USER)

## Access and schema check

Each line selects one row from the table itself, because a table can appear in the catalogue and still refuse a query. It also records every column name, so the download queries are written against the real schema rather than from memory.

In [ ]:
import pandas as pd

CHECKS = [
    # (label, table, optional coverage query)
    ("CRSP daily (CIZ)", "crsp.dsf_v2",
     "select min(dlycaldt) as first, max(dlycaldt) as last from crsp.dsf_v2 where permno = 14593"),
    ("CRSP names (CIZ)", "crsp.stocknames_v2", None),
    ("CRSP delistings (CIZ)", "crsp.stkdelists", None),
    ("CCM link history", "crsp.ccmxpf_lnkhist",
     "select linktype, linkprim, count(*) as n from crsp.ccmxpf_lnkhist group by 1, 2 order by 3 desc"),
    ("Fama-French daily factors", "ff.factors_daily",
     "select min(date) as first, max(date) as last from ff.factors_daily"),
    ("I/B/E/S actuals, unadjusted", "ibes.actu_epsus",
     "select pdicity, measure, count(*) as n, max(anndats) as last from ibes.actu_epsus "
     "where anndats >= '2024-01-01' group by 1, 2 order by 3 desc"),
    ("I/B/E/S detail, unadjusted", "ibes.detu_epsus",
     "select measure, fpi, count(*) as n, max(anndats) as last from ibes.detu_epsus "
     "where anndats >= '2025-01-01' group by 1, 2 order by 3 desc"),
    ("I/B/E/S guidance", "ibes.det_guidance",
     "select measure, count(*) as n, max(anndats) as last from ibes.det_guidance "
     "where anndats >= '2025-01-01' group by 1 order by 2 desc"),
    ("I/B/E/S to CRSP link", "wrdsapps.ibcrsphist",
     "select score, count(*) as n from wrdsapps.ibcrsphist group by 1 order by 1"),
    ("SEC CIK to GVKEY link", "wrdssec.wciklink_gvkey", None),
    ("Compustat quarterly (GAAP EPS check)", "comp.fundq",
     "select max(datadate) as last from comp.fundq"),
]


def rollback():
    # A failed query leaves the connection in an aborted transaction;
    # without this, every later check would fail too.
    try:
        db.connection.rollback()
    except Exception:
        pass


report = {"env": env, "user": WRDS_USER, "tables": {}}
for label, table, cov_sql in CHECKS:
    entry = {"label": label}
    try:
        one = db.raw_sql(f"select * from {table} limit 1")
        entry["access"] = "ok"
        entry["columns"] = list(one.columns)
    except Exception as e:
        rollback()
        entry["access"] = "DENIED_OR_MISSING"
        entry["error"] = str(e).splitlines()[0][:200]
    if cov_sql and entry["access"] == "ok":
        try:
            entry["coverage"] = db.raw_sql(cov_sql).astype(str).to_dict("records")
        except Exception as e:
            rollback()
            entry["coverage_error"] = str(e).splitlines()[0][:200]
    report["tables"][table] = entry
    print(f"{entry['access']:<18} {table:<26} {label}")

In [ ]:
# Table names we may need but have not confirmed (revenue estimates,
# stopped estimates, other CIZ files). Listing them lets us pick the right ones.
def tables_like(library, *patterns):
    try:
        names = db.list_tables(library=library)
    except Exception as e:
        rollback()
        return [f"error: {str(e).splitlines()[0][:120]}"]
    return sorted(n for n in names if any(p in n for p in patterns))


report["discovery"] = {
    "ibes": tables_like("ibes", "epsus", "xeps", "guid", "stop", "sum", "act", "det"),
    "crsp": tables_like("crsp", "_v2", "delist", "names", "ccm"),
    "ff": tables_like("ff", "factor"),
}

out = PROJECT_DIR / "logs" / f"access_report_{WRDS_USER}_{dt.date.today()}.json"
out.write_text(json.dumps(report, indent=2, default=str))

# Compact printout: copy everything from here down and send it back.
print("=" * 70)
print(json.dumps(env))
for table, e in report["tables"].items():
    print(f"\n{table}: {e['access']}")
    if "columns" in e:
        print("  columns:", ", ".join(e["columns"]))
    if "coverage" in e:
        print("  coverage:", e["coverage"][:8])
    if "error" in e or "coverage_error" in e:
        print("  error:", e.get("error") or e.get("coverage_error"))
for lib, names in report["discovery"].items():
    print(f"\n{lib} tables ({len(names)}):", ", ".join(names))
print("\nsaved:", out)

## Trading calendar

Trading days come from CRSP itself, cross-checked against the NYSE calendar, which also supplies early closes (13:00) and the sessions after CRSP's last date. Both "days in one but not the other" counts should be 0.

In [ ]:
from common import build_trading_calendar

cal = build_trading_calendar(db)
cal.to_csv(PROJECT_DIR / "data" / "trading_calendar.csv", index=False)
print(cal.tail(3).to_string())

## Provisional release clock on the real calendar

Same rule as before: wire time first, I/B/E/S only as the second source (or a development stand-in), nothing from the EDGAR submissions JSON. Expected: about 115,976 events, about 79,000 with two sources agreeing.

In [ ]:
from common import load_trading_calendar
from clock import build_provisional_clock, validate_clock, summarise

cal = load_trading_calendar(PROJECT_DIR / "data" / "trading_calendar.csv")
clk = build_provisional_clock(str(MAHAUT_DATA / "three_source_timing.csv"),
                              str(MAHAUT_DATA / "event_table.csv"), cal)
validate_clock(clk, cal)
print(summarise(clk).to_string())
clk.to_csv(PROJECT_DIR / "data" / "clock_provisional.csv", index=False)
print("saved:", PROJECT_DIR / "data" / "clock_provisional.csv")

In [14]:
db.close()